# Game Data Cleaning
Tujuan : Membersihkan data mengenai game berupa rating dan tahun rilis.

## 1. Baca dan Temukan Kesalahan

### 1.1 Baca Data

In [457]:
import pandas as pd

df = pd.read_csv("games_dirty.csv")

In [458]:
df.head()

,title,rating,release
0,The Witcher 3: Wild Hunt,9.2,2015
1,The Witcher 3: Wild Hunt,9.2,2015
2,Minecraft,NaN,2011
3,Cyberpunk 2077,7.0,NaN
4,Elden Ring,9.5,2022-02-25


In [459]:
df.tail()

,title,rating,release
45,Big Rigs: Over the Road Racing,0.4,2003
46,Left 4 Dead 2,8.9,2009
47,Half-Life 2,9.7,2004
48,Portal 2,9.8,2011
49,Left 4 Dead 2,8.9,2009


In [460]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    50 non-null     object
 1   rating   48 non-null     object
 2   release  47 non-null     object
dtypes: object(3)
memory usage: 1.3+ KB


### 1.2 Cek Kekosongan Data

In [461]:
data_kosong = df.isnull().sum()
print("Jumlah data yang kosong : ", data_kosong)

Jumlah data yang kosong :  title      0
rating     2
release    3
dtype: int64


Terdapat data kosong, yaitu rating dan release.
Keputusan : 
- Hapus data jika release kosong (karena tidak bisa kita asal menambahkan tahun rilis).
- Tambahkan average rating pada data rating yang kosong.

### 1.3 Cek Duplikat

In [462]:
data_duplikat = df.duplicated().sum()
print("Jumlah data duplikat : ", data_duplikat)

Jumlah data duplikat :  3


Terdapat tiga data yang duplikat. Keputusan :
- Hapus data duplikat.

### 1.4 Cek Nilai Aneh

In [463]:
# Cek data rating
df["rating"].unique()

array(['9.2', nan, '7.0', '9.5', '9.6', '0.5', '1.5', 'LIMA', '8.0',
       '9.3', '8.5', '9.4', '8.8', '7.2', '5.2', '8.9', '9.1', '6.5',
       '7.5', '9.7', '8.7', '9.0', '5.5', '7.1', '4.8', '1.3', '1.2',
       '0.4', '9.8'], dtype=object)

Disini terlihat ada nilai aneh seperti "LIMA" dan kekosongan data yang harus diperbaiki.
List Kesalahan :
- Kosong/NaN
- "LIMA"

In [464]:
# Cek data release
df["release"].unique()

array(['2015', '2011', nan, '2022-02-25', '2022', '2013s', '1982', '1999',
       '2009', '2.020', '2020', '2013', '11', '2018', '2016', '2006',
       '2024', '2017', '2023', '2019', '21', '2010', '2003', '2004'],
      dtype=object)

Pada kolom release juga terlihat ketidakkonsistenan nilai.
List Kesalahan :
- Kosong/NaN
- 2022-02-25
- 2013s
- 2.020
- 11
- 21

## 2. Perbaikan

### 2.1 Normalisasi Data

In [465]:
# Semua judul adalah string
df["title"] = df["title"].astype(str).str.strip().str.title()

### 2.2 Fix Rating Aneh

In [466]:
# ganti "LIMA"
df["rating"] = df["rating"].replace("LIMA", 5)

In [467]:
# Ubah rating menjadi angka (float)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

In [468]:
# Isi rating kosong dengan rata-rata
df["rating"] = df["rating"].fillna(df["rating"].mean())

### 2.3 Fix Release Aneh

In [490]:
# Hapus jika release kosong
df = df.dropna(subset="release")

In [491]:
# jadikan string dulu
df["release"] = df["release"].astype(str)

In [492]:
# ambil 4 karakter paling awal
df["release"] = df["release"].str.extract(r'(\d{4})')

In [493]:
# ubah jadi numeric, jika eror maka hapus
df["release"] = pd.to_numeric(df["release"], errors="coerce")

### 2.4 Hapus Duplikat

In [494]:
df = df.drop_duplicates()

## 3. Cek Kebersihan

In [495]:
kosong = df.isnull().sum()
print(kosong)

title      0
rating     0
release    0
dtype: int64


In [496]:
df

,title,rating,release
0,The Witcher 3: Wild Hunt,9.20000,2015
2,Minecraft,7.60625,2011
4,Elden Ring,9.50000,2022
6,Grand Theft Auto V,9.60000,2013
7,E.T. The Extra-Terrestrial,0.50000,1982
8,Superman 64,1.50000,1999
9,League Of Legends,5.00000,2009
11,The Last Of Us Part Ii,9.30000,2020
14,Dota 2,8.50000,2013
16,Terraria,8.80000,2011


## 4. Simpan

In [497]:
df.to_csv("games_clean.csv", index=False)